<a href="https://colab.research.google.com/github/minhaz-engg/Time-Series-try/blob/main/neuralforecast__MLPMultivariate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install neuralforecast

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from neuralforecast import NeuralForecast
from neuralforecast.models import MLPMultivariate
from neuralforecast.losses.pytorch import MAE

# ==========================================
# 1. Ingestion & Topological Restructuring
# ==========================================
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/avocado.csv"
df = pd.read_csv(url, index_col=0)

# Establish Canonical Schema
df['ds'] = pd.to_datetime(df['Date'])
df['y'] = df['AveragePrice']
# Create a bifurcated unique identifier: Geography + Product Topology
df['unique_id'] = df['region'] + "_" + df['type']

# Isolate a dense, representative subset to define the multivariate tensor
target_regions = ['California', 'NewYork', 'TotalUS']
df = df[df['region'].isin(target_regions)].copy()

# ==========================================
# 2. Exogenous Engineering (First Principles)
# ==========================================
# Transmute discrete time into continuous cyclical vectors
df['month'] = df['ds'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')

# Finalize Dynamic Temporal Matrix (Y_df)
Y_df = df[['unique_id', 'ds', 'y', 'month_sin', 'month_cos']]
Y_df = Y_df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Forge the Immutable Static Substrate (static_df)
static_df = df[['unique_id', 'type', 'region']].drop_duplicates().reset_index(drop=True)
# Binary Encoding for product variant
static_df['is_organic'] = (static_df['type'] == 'organic').astype(np.float32)
# Integer mapping for distinct geographical matrices
region_map = {reg: i for i, reg in enumerate(target_regions)}
static_df['region_enc'] = static_df['region'].map(region_map).astype(np.float32)
static_df = static_df[['unique_id', 'is_organic', 'region_enc']]

# ==========================================
# 3. Temporal Bifurcation (Train/Test)
# ==========================================
# The avocado data is sampled weekly ('W'). We extrapolate 12 weeks (approx 1 quarter)
horizon = 12
global_max_date = Y_df['ds'].max()
cutoff_date = global_max_date - pd.Timedelta(weeks=horizon)

Y_train_df = Y_df[Y_df['ds'] <= cutoff_date].reset_index(drop=True)
Y_test_df = Y_df[Y_df['ds'] > cutoff_date].reset_index(drop=True)

# ==========================================
# 4. Multivariate MLP Initialization
# ==========================================
# N_series = 3 regions * 2 types = 6 concurrent sequence trajectories
n_concurrent_series = len(Y_train_df['unique_id'].unique())

model = MLPMultivariate(
    h=horizon,
    input_size=52,             # Receptive field: ingest a full 52-week annual cycle
    n_series=n_concurrent_series,
    futr_exog_list=['month_sin', 'month_cos'], # Deterministic cyclic anchors
    stat_exog_list=['is_organic', 'region_enc'], # Immutable entity embeddings
    loss=MAE(),                # L1 norm to penalize ephemeral outlier spikes linearly
    scaler_type='robust',      # Homogenize disparate regional volume magnitudes
    learning_rate=1e-3,
    max_steps=300,
    val_check_steps=10,
    early_stop_patience_steps=3 # Mitigate paradigm overfitting
)

fcst = NeuralForecast(models=[model], freq='W') # 'W' defines weekly cadence

# Execution of Gradient Optimization
fcst.fit(df=Y_train_df, static_df=static_df, val_size=horizon)

# Inference projection
forecasts = fcst.predict(futr_df=Y_test_df)

# ==========================================
# 5. Advanced Empirical Visualization
# ==========================================
Y_hat_df = forecasts.reset_index(drop=False)
merged_plot_df = pd.merge(Y_test_df, Y_hat_df, on=['unique_id', 'ds'], how='inner')

# Configure sophisticated visual paradigm
sns.set_theme(style="darkgrid", context="paper")
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
axes = axes.flatten()

unique_ids = Y_train_df['unique_id'].unique()

for idx, uid in enumerate(unique_ids):
    ax = axes[idx]

    # Extract historical momentum (last 52 weeks of train data for context)
    train_subset = Y_train_df[Y_train_df['unique_id'] == uid].tail(52)
    test_subset = merged_plot_df[merged_plot_df['unique_id'] == uid]

    # Plot true historical trajectory
    ax.plot(train_subset['ds'], train_subset['y'], color='black', alpha=0.6, label='Historical Truth')

    # Plot true future trajectory
    ax.plot(test_subset['ds'], test_subset['y'], color='black', linestyle='--', linewidth=2, label='Future Truth')

    # Plot network extrapolation
    ax.plot(test_subset['ds'], test_subset['MLPMultivariate'], color='crimson', linewidth=2.5, label='MLP Projection')

    # Implied error boundary (aesthetic heuristic mapping)
    ax.fill_between(test_subset['ds'],
                    test_subset['MLPMultivariate'] * 0.95,
                    test_subset['MLPMultivariate'] * 1.05,
                    color='crimson', alpha=0.15)

    ax.set_title(f"Joint Projection Entity: {uid}", fontweight='bold')
    ax.set_ylabel("Average Price (USD)")

    if idx == 0:
        ax.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.suptitle("MLPMultivariate: Simultaneous Multidimensional Tensor Projections", y=1.02, fontsize=16, fontweight='bold')
plt.show()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ mlp          │ ModuleList    │  2.2 M │ train │     0 │
│ 4 │ out          │ Linear        │ 73.8 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.2 M                                                                                                
Total estimated model params size (MB): 8                                                                          
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()